# Notebook 02 — Mapa base interactivo con `folium`

Construimos un mapa `folium` centrado en la Sabana de Bogotá con tiles de OpenStreetMap y los plugins `MiniMap`, `MousePosition` y `Fullscreen`. Se guarda como HTML y como captura estática con Playwright.

**Salidas**: `media/02_folium_basemap.html`, `media/02_folium_basemap.png`

In [1]:
from pathlib import Path
import os
import asyncio
import folium
from folium.plugins import MiniMap, MousePosition, Fullscreen

def find_taller_root(start: Path, marker: str = "python/data/bogota_sabana_sentinel2.tif") -> Path:
    p = start.resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró {marker} desde {start}")

ROOT = find_taller_root(Path.cwd())
os.chdir(ROOT)
MEDIA = ROOT / "media"
MEDIA.mkdir(exist_ok=True)

CENTER = [4.66, -74.07]
ZOOM = 10

In [2]:
m = folium.Map(
    location=CENTER,
    zoom_start=ZOOM,
    tiles="OpenStreetMap",
    control_scale=True,
    width="100%",
    height="100%",
)

MiniMap(tile_layer="OpenStreetMap", position="bottomright", width=180, height=180, zoom_level_offset=-5).add_to(m)
MousePosition(position="bottomleft", separator=" , ", prefix="lat,lon:", num_digits=4).add_to(m)
Fullscreen(position="topleft").add_to(m)

folium.Marker(
    location=CENTER,
    popup="<b>Centro del AOI</b><br>Sabana de Bogotá",
    tooltip="Click para más info",
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(m)

folium.Circle(
    radius=5000,
    location=CENTER,
    color="crimson",
    fill=True,
    fill_color="crimson",
    fill_opacity=0.1,
    popup="Buffer 5 km",
).add_to(m)

m

In [3]:
html_path = MEDIA / "02_folium_basemap.html"
m.save(str(html_path))
print(f"HTML guardado: {html_path} ({html_path.stat().st_size/1024:.1f} KB)")

HTML guardado: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/02_folium_basemap.html (8.4 KB)


In [4]:
png_path = MEDIA / "02_folium_basemap.png"
from playwright.async_api import async_playwright

async def _screenshot():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={"width": 1200, "height": 800})
        page = await context.new_page()
        await page.goto(f"file://{html_path.resolve()}")
        await page.wait_for_load_state("networkidle")
        await page.wait_for_timeout(2000)
        await page.screenshot(path=str(png_path), full_page=False)
        await browser.close()

await _screenshot()
print(f"PNG guardado: {png_path} ({png_path.stat().st_size/1024:.1f} KB)")

PNG guardado: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/02_folium_basemap.png (954.8 KB)


## Conclusiones

- `folium.Map` permite crear un mapa interactivo centrado en la Sabana de Bogotá con un marcador, un círculo buffer de 5 km, y un minimapa.
- `LayerControl`, `MiniMap`, `MousePosition` y `Fullscreen` son plugins que enriquecen la interacción.
- En el notebook 05 se combinarán estas piezas con el raster NDVI y las capas vectoriales.